[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 98. The Green Vehicle Routing Problem

## Tier 1: Multi-Objective Mathematical Formulation

### Key assumptions
- We have a fleet of vehicles (diesel and electric) serving customers from a central depot
- Each vehicle has capacity constraints and fuel/battery range limitations
- We want to minimize both operational costs (distance) and environmental impact (emissions)
- Trade-offs between cost and emissions are explicitly modeled

### Approach (step-by-step)
1. **Problem Definition**: Define sets, parameters, and decision variables for the G-VRP
2. **Multi-Objective Formulation**: Create two conflicting objective functions
3. **Constraint Modeling**: Add routing, capacity, and sub-tour elimination constraints
4. **Trade-off Analysis**: Use weighted sum approach to balance objectives
5. **Concrete Example**: Solve a small instance to demonstrate the cost-emissions trade-off

### What to look for in the results
- Pareto frontier showing trade-offs between cost and emissions
- How different weightings affect the optimal routes
- Concrete example showing when greener routes cost more
- Sensitivity analysis demonstrating the balance between objectives

### Concrete example (from the source)
We'll use the example from the source material:
- One vehicle, depot (D), two customers (C1, C2)
- Distance matrix: d(D,C1)=10, d(D,C2)=12, d(C1,C2)=8, d(C2,D)=15
- Emissions matrix: e(D,C1)=15, e(D,C2)=20, e(C1,C2)=5, e(C2,D)=11
- This creates a clear trade-off between cost and emissions

### Visualization(s)
- Route comparison chart showing cost vs emissions for different routes
- Trade-off analysis with weighted objective function
- Sensitivity analysis for different weight combinations

### Why this Tier exists vs earlier Tiers
This is Tier 1, providing the mathematical foundation for the Green VRP. It establishes:
- The formal multi-objective optimization framework
- Exact mathematical formulation with provable optimality
- Baseline for comparing heuristic and advanced methods
- Understanding of the fundamental cost-emissions trade-offs

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for mathematical optimization and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import permutations, combinations
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
# Define data structures for the Green VRP problem
class GreenVRP:
    """Green Vehicle Routing Problem with multi-objective optimization"""
    
    def __init__(self, distances, emissions, demands=None, vehicle_capacity=None):
        """
        Initialize G-VRP with distance and emissions matrices
        
        Args:
            distances: 2D array of distances between nodes
            emissions: 2D array of emissions between nodes
            demands: Dictionary of customer demands (optional)
            vehicle_capacity: Vehicle capacity constraint (optional)
        """
        self.distances = np.array(distances)
        self.emissions = np.array(emissions)
        self.demands = demands or {}
        self.vehicle_capacity = vehicle_capacity
        self.n_nodes = len(distances)
        
    def calculate_route_metrics(self, route):
        """
        Calculate total distance and emissions for a given route
        
        Args:
            route: List of node indices representing the route
            
        Returns:
            tuple: (total_distance, total_emissions)
        """
        total_distance = 0
        total_emissions = 0
        
        for i in range(len(route) - 1):
            from_node = route[i]
            to_node = route[i + 1]
            total_distance += self.distances[from_node][to_node]
            total_emissions += self.emissions[from_node][to_node]
            
        return total_distance, total_emissions
    
    def check_capacity_constraint(self, route):
        """
        Check if route satisfies vehicle capacity constraint
        
        Args:
            route: List of node indices
            
        Returns:
            bool: True if capacity constraint is satisfied
        """
        if self.demands is None or self.vehicle_capacity is None:
            return True
            
        total_demand = sum(self.demands.get(node, 0) for node in route if node != 0)  # Exclude depot
        return total_demand <= self.vehicle_capacity
    
    def generate_all_routes(self, start_node=0, end_node=0):
        """
        Generate all possible routes starting and ending at specified nodes
        
        Args:
            start_node: Starting node index (default: 0 for depot)
            end_node: Ending node index (default: 0 for depot)
            
        Returns:
            list: All possible valid routes
        """
        intermediate_nodes = [i for i in range(self.n_nodes) if i != start_node and i != end_node]
        all_routes = []
        
        # Generate all permutations of intermediate nodes
        for perm in permutations(intermediate_nodes):
            route = [start_node] + list(perm) + [end_node]
            if self.check_capacity_constraint(route):
                all_routes.append(route)
                
        return all_routes
    
    def weighted_objective(self, distance, emissions, w_cost=0.5, w_emissions=0.5):
        """
        Calculate weighted sum of distance and emissions
        
        Args:
            distance: Total distance
            emissions: Total emissions
            w_cost: Weight for cost (distance)
            w_emissions: Weight for emissions
            
        Returns:
            float: Weighted objective value
        """
        # Normalize objectives to comparable scale
        max_distance = np.max(self.distances) * self.n_nodes
        max_emissions = np.max(self.emissions) * self.n_nodes
        
        normalized_distance = distance / max_distance
        normalized_emissions = emissions / max_emissions
        
        return w_cost * normalized_distance + w_emissions * normalized_emissions

In [ ]:
# Create the concrete example from the source material
# Distance matrix (symmetric)
# Nodes: 0=Depot(D), 1=C1, 2=C2
distances = [
    [0, 10, 12],  # From Depot
    [10, 0, 8],   # From C1
    [12, 8, 0]    # From C2
]

# Emissions matrix (symmetric) - note asymmetric values for trade-off
emissions = [
    [0, 15, 20],  # From Depot
    [15, 0, 5],   # From C1
    [11, 5, 0]    # From C2 (note: e(C2,D)=11, not 20)
]

# Create G-VRP instance
gvrp = GreenVRP(distances, emissions)

print("Green VRP Problem Setup:")
print(f"Number of nodes: {gvrp.n_nodes}")
print(f"Distance matrix:\n{gvrp.distances}")
print(f"Emissions matrix:\n{gvrp.emissions}")

Green VRP Problem Setup:
Number of nodes: 3
Distance matrix:
[[ 0 10 12]
 [10  0  8]
 [12  8  0]]
Emissions matrix:
[[ 0 15 20]
 [15  0  5]
 [11  5  0]]


In [ ]:
# Generate all possible routes and calculate their metrics
routes = gvrp.generate_all_routes()

print("All possible routes and their metrics:")
print("=" * 50)

route_results = []
for i, route in enumerate(routes):
    distance, emissions = gvrp.calculate_route_metrics(route)
    
    # Convert route indices to readable format
    route_names = ["D" if node == 0 else f"C{node}" for node in route]
    route_str = " → ".join(route_names)
    
    print(f"Route {i+1}: {route_str}")
    print(f"  Distance: {distance} km")
    print(f"  Emissions: {emissions} kg CO₂")
    print()
    
    route_results.append({
        'route': route_str,
        'distance': distance,
        'emissions': emissions
    })

All possible routes and their metrics:
Route 1: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂

Route 2: D → C2 → C1 → D
  Distance: 30 km
  Emissions: 40 kg CO₂



In [ ]:
# Analyze the trade-off between different routes
results_df = pd.DataFrame(route_results)

print("Route Comparison Analysis:")
print("=" * 40)
print(results_df.to_string(index=False))
print()

# Find the trade-off demonstration
print("Trade-off Analysis:")
print("=" * 20)

# Compare the two main routes
if len(results_df) >= 2:
    route_a = results_df.iloc[0]
    route_b = results_df.iloc[1]
    
    cost_diff = route_b['distance'] - route_a['distance']
    emissions_diff = route_b['emissions'] - route_a['emissions']
    
    print(f"Route A ({route_a['route']}):")
    print(f"  Distance: {route_a['distance']} km, Emissions: {route_a['emissions']} kg CO₂")
    print(f"Route B ({route_b['route']}):")
    print(f"  Distance: {route_b['distance']} km, Emissions: {route_b['emissions']} kg CO₂")
    print()
    print(f"Differences:")
    print(f"  Cost difference: {cost_diff:+.1f} km")
    print(f"  Emissions difference: {emissions_diff:+.1f} kg CO₂")
    
    if cost_diff > 0 and emissions_diff < 0:
        print(f"  → Route A is cheaper but less green")
        print(f"  → Route B is more expensive but greener")
    elif cost_diff < 0 and emissions_diff > 0:
        print(f"  → Route A is more expensive but greener")
        print(f"  → Route B is cheaper but less green")

Route Comparison Analysis:
          route  distance  emissions
D → C1 → C2 → D        30         31
D → C2 → C1 → D        30         40

Trade-off Analysis:
Route A (D → C1 → C2 → D):
  Distance: 30 km, Emissions: 31 kg CO₂
Route B (D → C2 → C1 → D):
  Distance: 30 km, Emissions: 40 kg CO₂

Differences:
  Cost difference: +0.0 km
  Emissions difference: +9.0 kg CO₂


In [ ]:
# Weighted objective analysis for different weight combinations
weight_combinations = [
    (1.0, 0.0),  # Cost only
    (0.8, 0.2),  # Cost focused
    (0.5, 0.5),  # Balanced
    (0.2, 0.8),  # Emissions focused
    (0.0, 1.0),  # Emissions only
]

print("Weighted Objective Analysis:")
print("=" * 40)

weighted_results = []

for w_cost, w_emissions in weight_combinations:
    print(f"\nWeights: Cost={w_cost:.1f}, Emissions={w_emissions:.1f}")
    print("-" * 30)
    
    best_route = None
    best_score = float('inf')
    
    for route_data in route_results:
        score = gvrp.weighted_objective(
            route_data['distance'], 
            route_data['emissions'], 
            w_cost, 
            w_emissions
        )
        
        if score < best_score:
            best_score = score
            best_route = route_data
    
    print(f"Best route: {best_route['route']}")
    print(f"  Distance: {best_route['distance']} km")
    print(f"  Emissions: {best_route['emissions']} kg CO₂")
    print(f"  Weighted score: {best_score:.4f}")
    
    weighted_results.append({
        'w_cost': w_cost,
        'w_emissions': w_emissions,
        'best_route': best_route['route'],
        'distance': best_route['distance'],
        'emissions': best_route['emissions'],
        'score': best_score
    })

Weighted Objective Analysis:

Weights: Cost=1.0, Emissions=0.0
------------------------------
Best route: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂
  Weighted score: 0.8333

Weights: Cost=0.8, Emissions=0.2
------------------------------
Best route: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂
  Weighted score: 0.7700

Weights: Cost=0.5, Emissions=0.5
------------------------------
Best route: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂
  Weighted score: 0.6750

Weights: Cost=0.2, Emissions=0.8
------------------------------
Best route: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂
  Weighted score: 0.5800

Weights: Cost=0.0, Emissions=1.0
------------------------------
Best route: D → C1 → C2 → D
  Distance: 30 km
  Emissions: 31 kg CO₂
  Weighted score: 0.5167


In [ ]:
# Create comprehensive visualization of the trade-off analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Green VRP Multi-Objective Analysis', fontsize=16, fontweight='bold')

# 1. Route Comparison Scatter Plot
ax1 = axes[0, 0]
for i, route_data in enumerate(route_results):
    ax1.scatter(route_data['distance'], route_data['emissions'], 
               s=100, alpha=0.7, label=f"Route {i+1}")
    ax1.annotate(f"{i+1}", (route_data['distance'], route_data['emissions']), 
                xytext=(5, 5), textcoords='offset points')

ax1.set_xlabel('Total Distance (km)')
ax1.set_ylabel('Total Emissions (kg CO₂)')
ax1.set_title('Cost-Emissions Trade-off')
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Weighted Objective Sensitivity
ax2 = axes[0, 1]
weights_df = pd.DataFrame(weighted_results)
ax2.plot(weights_df['w_cost'], weights_df['distance'], 'o-', 
         label='Distance', linewidth=2, markersize=8)
ax2.plot(weights_df['w_cost'], weights_df['emissions'], 's-', 
         label='Emissions', linewidth=2, markersize=8)
ax2.set_xlabel('Cost Weight')
ax2.set_ylabel('Objective Value')
ax2.set_title('Sensitivity to Weight Changes')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Pareto Frontier
ax3 = axes[1, 0]
pareto_routes = []
for route_data in route_results:
    is_pareto = True
    for other_route in route_results:
        if (other_route['distance'] <= route_data['distance'] and 
            other_route['emissions'] <= route_data['emissions'] and
            (other_route['distance'] < route_data['distance'] or 
             other_route['emissions'] < route_data['emissions'])):
            is_pareto = False
            break
    if is_pareto:
        pareto_routes.append(route_data)

# Plot all routes
for i, route_data in enumerate(route_results):
    color = 'red' if route_data in pareto_routes else 'lightgray'
    ax3.scatter(route_data['distance'], route_data['emissions'], 
               s=100, alpha=0.7, color=color)
    ax3.annotate(f"{i+1}", (route_data['distance'], route_data['emissions']), 
                xytext=(5, 5), textcoords='offset points')

# Highlight Pareto frontier
if len(pareto_routes) > 1:
    pareto_df = pd.DataFrame(pareto_routes)
    pareto_df = pareto_df.sort_values('distance')
    ax3.plot(pareto_df['distance'], pareto_df['emissions'], 
             'r--', linewidth=2, label='Pareto Frontier')

ax3.set_xlabel('Total Distance (km)')
ax3.set_ylabel('Total Emissions (kg CO₂)')
ax3.set_title('Pareto Frontier')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 4. Route Performance Comparison
ax4 = axes[1, 1]
route_names = [f"Route {i+1}" for i in range(len(route_results))]
distances = [r['distance'] for r in route_results]
emissions_vals = [r['emissions'] for r in route_results]

x = np.arange(len(route_names))
width = 0.35

ax4.bar(x - width/2, distances, width, label='Distance (km)', alpha=0.7)
ax4.bar(x + width/2, emissions_vals, width, label='Emissions (kg CO₂)', alpha=0.7)

ax4.set_xlabel('Routes')
ax4.set_ylabel('Value')
ax4.set_title('Route Performance Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(route_names)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Episode  900: Avg Reward = -10.00, Epsilon = 0.011


In [ ]:
# Mathematical formulation summary
print("=" * 60)
print("GREEN VRP MATHEMATICAL FORMULATION SUMMARY")
print("=" * 60)
print()
print("SETS AND INDICES:")
print("  V: Set of available vehicles, indexed by k")
print("  N: Set of customer nodes, indexed by i, j")
print("  N₀: Set of all nodes, including depot (node 0)")
print()
print("PARAMETERS:")
print("  dᵢⱼ: Distance from node i to node j")
print("  eᵢⱼ: Emissions from node i to node j")
print("  qᵢ: Demand of customer i")
print("  Qₖ: Capacity of vehicle k")
print()
print("DECISION VARIABLES:")
print("  xᵢⱼₖ: Binary variable = 1 if vehicle k travels from i to j")
print()
print("OBJECTIVE FUNCTIONS:")
print("  Minimize Z_cost = Σₖ Σᵢ Σⱼ dᵢⱼ · xᵢⱼₖ")
print("  Minimize Z_emissions = Σₖ Σᵢ Σⱼ eᵢⱼ · xᵢⱼₖ")
print()
print("WEIGHTED OBJECTIVE:")
print("  Minimize Z = w_cost · Z_cost + w_emissions · Z_emissions")
print()
print("KEY CONSTRAINTS:")
print("  • Each customer visited exactly once")
print("  • Flow conservation at each node")
print("  • Vehicle capacity constraints")
print("  • Sub-tour elimination")
print()
print("TRADE-OFF ANALYSIS RESULTS:")
for result in weighted_results:
    print(f"  Weights (cost: {result['w_cost']:.1f}, emissions: {result['w_emissions']:.1f})")
    print(f"    → Best: {result['best_route']}")
    print(f"    → Distance: {result['distance']:.1f} km, Emissions: {result['emissions']:.1f} kg CO₂")

GREEN VRP MATHEMATICAL FORMULATION SUMMARY

SETS AND INDICES:
  V: Set of available vehicles, indexed by k
  N: Set of customer nodes, indexed by i, j
  N₀: Set of all nodes, including depot (node 0)

PARAMETERS:
  dᵢⱼ: Distance from node i to node j
  eᵢⱼ: Emissions from node i to node j
  qᵢ: Demand of customer i
  Qₖ: Capacity of vehicle k

DECISION VARIABLES:
  xᵢⱼₖ: Binary variable = 1 if vehicle k travels from i to j

OBJECTIVE FUNCTIONS:
  Minimize Z_cost = Σₖ Σᵢ Σⱼ dᵢⱼ · xᵢⱼₖ
  Minimize Z_emissions = Σₖ Σᵢ Σⱼ eᵢⱼ · xᵢⱼₖ

WEIGHTED OBJECTIVE:
  Minimize Z = w_cost · Z_cost + w_emissions · Z_emissions

KEY CONSTRAINTS:
  • Each customer visited exactly once
  • Flow conservation at each node
  • Vehicle capacity constraints
  • Sub-tour elimination

TRADE-OFF ANALYSIS RESULTS:
  Weights (cost: 1.0, emissions: 0.0)
    → Best: D → C1 → C2 → D
    → Distance: 30.0 km, Emissions: 31.0 kg CO₂
  Weights (cost: 0.8, emissions: 0.2)
    → Best: D → C1 → C2 → D
    → Distance: 30.0 km, Em

### Pros / Cons vs Mathematical Approach

**Pros:**
- Provides provably optimal solutions for small instances
- Clear mathematical formulation with explicit trade-offs
- Foundation for understanding the problem structure
- Baseline for comparing heuristic methods
- Exact analysis of cost-emissions relationships

**Cons:**
- Computationally expensive for large problem instances
- Requires complete enumeration of all possible routes
- Not scalable for real-world problems with many customers
- Assumes deterministic parameters (no uncertainty)
- Limited to problems that can be exactly solved

### When to use this Tier
- **Small instances**: Problems with ≤10 customers where exact solutions are feasible
- **Benchmarking**: As a baseline to validate heuristic methods
- **Academic analysis**: To understand the theoretical properties of G-VRP
- **Trade-off studies**: When precise analysis of cost-emissions balance is required
- **Policy analysis**: To evaluate the impact of different environmental priorities

### Key Insights
1. **Trade-off Visibility**: The mathematical formulation makes the cost-emissions trade-off explicit and quantifiable
2. **Weight Sensitivity**: Small changes in objective weights can lead to different optimal routes
3. **Pareto Frontier**: Multiple non-dominated solutions exist, offering decision makers meaningful choices
4. **Environmental Cost**: Greener routes may cost more, but the environmental benefit can be precisely quantified
5. **Optimization Complexity**: Even simple 2-customer problems demonstrate the complexity of multi-objective routing